In [39]:
import xgboost as xgb
import pandas as pd
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import cross_val_score, KFold, train_test_split

In [13]:
data_path = '../dataset_1_pixels.csv'

df = pd.read_csv(data_path)

In [14]:
X, y = df.drop(['pm25_change', 'lon', 'lat', 'date'], axis=1), df['pm25_change']

train_ratio = 0.8
test_ratio = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_ratio, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [43]:
def train_and_eval(model, X_train, y_train, X_test, y_test):
    print('-----------------------------------------------------------------')
    print(f'Evaluating model {model}...')
    model.fit(X_train, y_train)

    y_preds = model.predict(X_test)

    mse = mean_squared_error(y_test, y_preds)
    rmse = root_mean_squared_error(y_test, y_preds)
    r2 = r2_score(y_test, y_preds)

    print(f"MSE: {mse}")
    print(f"RMSE: {rmse}")
    print(f"R^2: {r2}")
    print('-----------------------------------------------------------------')

In [48]:
dummy = DummyRegressor(strategy='mean')
ols = LinearRegression()

train_and_eval(dummy, X_train, y_train, X_test, y_test)
train_and_eval(ols, X_train, y_train, X_test, y_test)

-----------------------------------------------------------------
Evaluating model DummyRegressor()...
MSE: 65.77147620194688
RMSE: 8.10996154133587
R^2: -9.827466396217233e-09
-----------------------------------------------------------------
-----------------------------------------------------------------
Evaluating model LinearRegression()...
MSE: 59.02164683909512
RMSE: 7.682554707849149
R^2: 0.1026254719005184
-----------------------------------------------------------------


In [49]:
print('-----------------------------------------------------------------')
model = xgb.XGBRegressor(tree_method="hist", early_stopping_rounds=10)
print(f'Evaluating model {model}...')
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_preds = model.predict(X_test)

mse = mean_squared_error(y_test, y_preds)
rmse = root_mean_squared_error(y_test, y_preds)
r2 = r2_score(y_test, y_preds)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R^2: {r2}")
print('-----------------------------------------------------------------')

-----------------------------------------------------------------
Evaluating model XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=10,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)...
MSE: 44.61008451969657
RMSE: 6.6790781189994
R^2: 0.3217411629757492
-----------------------------------------------------------------
